# March Madness 2026 (Women's) - Simulator

**Quick start:**
1. Make sure `data/W_trained_params.json` exists (run `W_train.ipynb` first).
2. Load trained params and feature scaler.
3. Load the 2026 women's Kaggle season stats (no KenPom login required).
4. Run the remaining cells to simulate the bracket.

All model logic lives in `marchmadness/`. This notebook is a thin runner around the package.

**No KenPom required** — all stats come from Kaggle `WRegularSeasonDetailedResults.csv`.

In [1]:
# Cell 1 - Imports
import os
import sys

sys.path.insert(0, ".")

from marchmadness import (
    load_scaler,
    normalize_team_features,
    load_kaggle_season_stats,
    build_kaggle_schedule_features,
    build_team_features,
    ModelParams,
    load_params,
    build_full_bracket,
    simulate_full_bracket,
    simulate_tournament_round_by_round,
    print_first_round_matchups,
    print_champion_counts,
    print_full_tournament,
)

In [2]:
# Cell 2 - Load parameters

params = load_params("data/W_trained_params.json")
print("Loaded trained params from data/W_trained_params.json")

print(f"  margin_scale={params.margin_scale:.2f}  shock_df={params.shock_df:.2f}  shock_scale={params.shock_scale:.4f}")
print(f"  luck_weight={params.luck_weight:.4f}  recent_form_weight={params.recent_form_weight:.4f}")
print(f"  seed_prior_weight={params.seed_prior_weight:.4f}")
print(f"  w_efg={params.w_efg:.4f}  w_to={params.w_to:.4f}  w_orb={params.w_orb:.4f}  w_ftr={params.w_ftr:.4f}")

N_SIM_FULL = 10_000
N_SIM_ROUND = 1_000
SEED = 142

# Extra simulation-only late-round softening for the women's bracket.
# The trained global tau is kept for early rounds; later rounds are pushed to
# higher effective taus because elite-vs-elite matchups were still too sharp.
W_TARGET_TAU_BY_ROUND = {
    1: float(params.temperature),
    2: max(float(params.temperature), 2.00),
    3: max(float(params.temperature), 2.20),
    4: max(float(params.temperature), 2.40),
    5: max(float(params.temperature), 2.55),
    6: max(float(params.temperature), 2.55),
}
W_ROUND_TEMPERATURE_MULTIPLIERS = {
    round_number: target_tau / max(float(params.temperature), 0.1)
    for round_number, target_tau in W_TARGET_TAU_BY_ROUND.items()
}
print("Women's effective tau by round:")
for round_number, label in [(1, "Round of 64"), (2, "Round of 32"), (3, "Sweet 16"), (4, "Elite 8"), (5, "Final Four"), (6, "Championship")]:
    print(f"  {label:<13} -> tau={W_TARGET_TAU_BY_ROUND[round_number]:.3f}")

# Load feature scaler (produced by W_train.ipynb)
_scaler_path = "data/W_feature_scaler.json"
if os.path.exists(_scaler_path):
    feature_scaler = load_scaler(_scaler_path)
    print(f"Loaded feature scaler from {_scaler_path}")
else:
    feature_scaler = None
    print(f"No scaler found at {_scaler_path} — raw feature values will be used.")


Loaded trained params from data/W_trained_params.json
  margin_scale=70.01  shock_df=5.01  shock_scale=0.0769
  luck_weight=0.0000  recent_form_weight=0.0000
  seed_prior_weight=0.2091
  w_efg=0.0140  w_to=-0.0020  w_orb=0.0156  w_ftr=-0.0082
Women's effective tau by round:
  Round of 64   -> tau=0.970
  Round of 32   -> tau=2.000
  Sweet 16      -> tau=2.200
  Elite 8       -> tau=2.400
  Final Four    -> tau=2.550
  Championship  -> tau=2.550
Loaded feature scaler from data/W_feature_scaler.json


In [3]:
# Cell 3 - Load 2026 women's season stats from Kaggle
#
# No browser/login required. Reads WRegularSeasonDetailedResults.csv.
# Normalize a small set of bracket-name aliases so feature keys line up with
# the hardcoded tournament field below.

SEASON_YEAR = 2026
W_BRACKET_NAME_MAP = {
    "WI Green Bay": "Green Bay",
}

kenpom_df = load_kaggle_season_stats([SEASON_YEAR], "data/kaggle", gender="W")[SEASON_YEAR].copy()
kenpom_df["Team"] = kenpom_df["Team"].replace(W_BRACKET_NAME_MAP)
print(f"Loaded {len(kenpom_df)} women's teams for {SEASON_YEAR}")


Loaded 363 women's teams for 2026


In [4]:
# Cell 4 - 2026 NCAA Women's Tournament bracket (hardcoded from official bracket)
#
# Region hosting sites:
#   South   -> Fort Worth 1
#   East    -> Sacramento 2
#   Midwest -> Sacramento 4
#   West    -> Fort Worth 3
#
# First Four results (picked by better regular-season record):
#   Southern Univ (19-13) def. Samford (16-18)  -> 16-seed Midwest
#   SF Austin (25-9)      def. Missouri St (22-12) -> 16-seed West
#   Arizona St (24-10)    def. Virginia (19-11)    -> 10-seed Midwest
#   Richmond (26-7)       def. Nebraska (18-12)    -> 11-seed East

first_four_teams = {}  # All First Four resolved

main_bracket_teams = {
    # SOUTH (Fort Worth 1)
    "Connecticut":    (1,  "South"),
    "UT San Antonio": (16, "South"),   # UTSA
    "Iowa St":        (8,  "South"),
    "Syracuse":       (9,  "South"),
    "Maryland":       (5,  "South"),
    "Murray St":      (12, "South"),
    "North Carolina": (4,  "South"),
    "W Illinois":     (13, "South"),
    "Notre Dame":     (6,  "South"),
    "Fairfield":      (11, "South"),
    "Ohio St":        (3,  "South"),
    "Howard":         (14, "South"),
    "Illinois":       (7,  "South"),
    "Colorado":       (10, "South"),
    "Vanderbilt":     (2,  "South"),
    "High Point":     (15, "South"),
    # EAST (Sacramento 2)
    "UCLA":           (1,  "East"),
    "Cal Baptist":    (16, "East"),
    "Oklahoma St":    (8,  "East"),
    "Princeton":      (9,  "East"),
    "Mississippi":    (5,  "East"),   # Ole Miss
    "Gonzaga":        (12, "East"),
    "Minnesota":      (4,  "East"),
    "Green Bay":      (13, "East"),
    "Baylor":         (6,  "East"),
    "Richmond":       (11, "East"),   # First Four winner (def. Nebraska 26-7)
    "Duke":           (3,  "East"),
    "Col Charleston": (14, "East"),
    "Texas Tech":     (7,  "East"),
    "Villanova":      (10, "East"),
    "LSU":            (2,  "East"),
    "Jacksonville":   (15, "East"),
    # MIDWEST (Sacramento 4)
    "South Carolina": (1,  "Midwest"),
    "Southern Univ":  (16, "Midwest"),  # First Four winner (def. Samford 19-13)
    "Clemson":        (8,  "Midwest"),
    "USC":            (9,  "Midwest"),
    "Michigan St":    (5,  "Midwest"),
    "Colorado St":    (12, "Midwest"),
    "Oklahoma":       (4,  "Midwest"),
    "Idaho":          (13, "Midwest"),
    "Washington":     (6,  "Midwest"),
    "S Dakota St":    (11, "Midwest"),
    "TCU":            (3,  "Midwest"),
    "UC San Diego":   (14, "Midwest"),
    "Georgia":        (7,  "Midwest"),
    "Arizona St":     (10, "Midwest"),  # First Four winner (def. Virginia 24-10)
    "Iowa":           (2,  "Midwest"),
    "F Dickinson":    (15, "Midwest"),
    # WEST (Fort Worth 3)
    "Texas":          (1,  "West"),
    "SF Austin":      (16, "West"),     # First Four winner (def. Missouri St 25-9)
    "Oregon":         (8,  "West"),
    "Virginia Tech":  (9,  "West"),
    "Kentucky":       (5,  "West"),
    "James Madison":  (12, "West"),
    "West Virginia":  (4,  "West"),
    "Miami OH":       (13, "West"),
    "Alabama":        (6,  "West"),
    "Rhode Island":   (11, "West"),
    "Louisville":     (3,  "West"),
    "Vermont":        (14, "West"),
    "NC State":       (7,  "West"),
    "Tennessee":      (10, "West"),
    "Michigan":       (2,  "West"),
    "Holy Cross":     (15, "West"),
}

print(f"Main bracket: {len(main_bracket_teams)} teams (should be 64)")


Main bracket: 64 teams (should be 64)


In [5]:
# Cell 5 - Build schedule features for 2026 bracket teams
#
# Derives recent_form and conf_tourney_wins from Kaggle regular-season results.
# Uses build_kaggle_schedule_features on the current season data.
# (This differs from the men's notebook which uses KenPom schedules.)

sf_all, ct_all = build_kaggle_schedule_features([SEASON_YEAR], "data/kaggle", gender="W")
schedule_features_map = {
    W_BRACKET_NAME_MAP.get(team, team): values
    for team, values in sf_all.get(SEASON_YEAR, {}).items()
}
conf_tourney_results = {
    W_BRACKET_NAME_MAP.get(team, team): wins
    for team, wins in ct_all.get(SEASON_YEAR, {}).items()
}

print(f"Built schedule features for {len(schedule_features_map)} teams.")


Built schedule features for 363 teams.


In [7]:
# Cell 6 - Build features + bracket

all_seeds = {**main_bracket_teams, **first_four_teams}

features = build_team_features(
    kenpom_df,
    seeds=all_seeds,
    schedule_features_map=schedule_features_map,
    conf_tourney_results=conf_tourney_results or None,
    kaggle_team_features=None,  # W model uses seed prior, not box-score features
)
print(f"Built features for {len(features)} teams.")

# Apply feature normalization (uses scaler fitted during W_train.ipynb)
if feature_scaler is not None:
    features = normalize_team_features(features, feature_scaler)
    print("Applied feature normalization.")


first_four_games, round_of_64 = build_full_bracket(main_bracket_teams, first_four_teams)
print(f"First Four: {len(first_four_games)} games | Round of 64: {len(round_of_64)} games")

print_first_round_matchups(round_of_64)

Built features for 64 teams.
Applied feature normalization.
First Four: 0 games | Round of 64: 32 games
Tournament Start

South Region
----------------------------------------
  Connecticut vs UT San Antonio
  Iowa St vs Syracuse
  Maryland vs Murray St
  North Carolina vs W Illinois
  Notre Dame vs Fairfield
  Ohio St vs Howard
  Illinois vs Colorado
  Vanderbilt vs High Point

East Region
----------------------------------------
  UCLA vs Cal Baptist
  Oklahoma St vs Princeton
  Mississippi vs Gonzaga
  Minnesota vs Green Bay
  Baylor vs Richmond
  Duke vs Col Charleston
  Texas Tech vs Villanova
  LSU vs Jacksonville

Midwest Region
----------------------------------------
  South Carolina vs Southern Univ
  Clemson vs USC
  Michigan St vs Colorado St
  Oklahoma vs Idaho
  Washington vs S Dakota St
  TCU vs UC San Diego
  Georgia vs Arizona St
  Iowa vs F Dickinson

West Region
----------------------------------------
  Texas vs SF Austin
  Oregon vs Virginia Tech
  Kentucky vs Jame

In [12]:
# Cell 7 - Monte Carlo: championship probabilities

champion_counts = simulate_full_bracket(
    first_four_games, round_of_64,
    features, params,
    n_sim=N_SIM_FULL,
    seed=SEED,
)

print_champion_counts(champion_counts, n_sim=N_SIM_FULL, top_n=12)


Simulation Progress: [##################################################] 100.0%

Champion Probabilities (Top 12)
   1. Connecticut                3697/10000   37.0%  ██████████████████
   2. UCLA                       2797/10000   28.0%  █████████████
   3. Texas                      1730/10000   17.3%  ████████
   4. South Carolina             1270/10000   12.7%  ██████
   5. LSU                         169/10000    1.7%  
   6. Michigan                    104/10000    1.0%  
   7. Iowa                         86/10000    0.9%  
   8. Duke                         40/10000    0.4%  
   9. Louisville                   25/10000    0.2%  
  10. Ohio St                      20/10000    0.2%  
  11. West Virginia                14/10000    0.1%  
  12. Vanderbilt                   13/10000    0.1%  


In [13]:
# Cell 7b - Champion concentration diagnostics
#
# Track these numbers before and after adding temperature calibration.
# Expected changes when tau > 1 is applied:
#   - champion entropy goes up (distribution flattens)
#   - top-1 title probability comes down materially
#   - # of teams with >1% chance increases

import numpy as np

print(f"Temperature (tau) in loaded params: {params.temperature:.4f}")
print()

total = sum(champion_counts.values())
probs_champ = {t: c / total for t, c in champion_counts.items()}
nonzero = {t: p for t, p in probs_champ.items() if p > 0}

# Entropy: lower = more concentrated, higher = more spread
entropy = -sum(p * np.log2(p) for p in nonzero.values())
max_entropy = np.log2(len(main_bracket_teams))
concentration = 1.0 - entropy / max_entropy   # 0=uniform, 1=one team wins always

top1 = max(probs_champ.values())
top3 = sum(sorted(probs_champ.values(), reverse=True)[:3])

teams_above_1pct  = sum(1 for p in probs_champ.values() if p >= 0.01)
teams_above_5pct  = sum(1 for p in probs_champ.values() if p >= 0.05)
teams_above_10pct = sum(1 for p in probs_champ.values() if p >= 0.10)

print("Champion distribution summary:")
print(f"  entropy            = {entropy:.2f} bits  (max = {max_entropy:.2f})")
print(f"  concentration      = {concentration:.3f}  (0=uniform, 1=all one team)")
print(f"  top-1 probability  = {top1:.1%}")
print(f"  top-3 combined     = {top3:.1%}")
print(f"  teams \u2265 1% chance  = {teams_above_1pct}")
print(f"  teams \u2265 5% chance  = {teams_above_5pct}")
print(f"  teams \u2265 10% chance = {teams_above_10pct}")

Temperature (tau) in loaded params: 0.9698

Champion distribution summary:
  entropy            = 2.22 bits  (max = 6.00)
  concentration      = 0.630  (0=uniform, 1=all one team)
  top-1 probability  = 37.0%
  top-3 combined     = 82.2%
  teams ≥ 1% chance  = 6
  teams ≥ 5% chance  = 4
  teams ≥ 10% chance = 4


In [14]:
# Cell 8 - Round-by-round: most likely bracket path

round_results, round_details = simulate_tournament_round_by_round(
    first_four_games, round_of_64,
    features, params,
    n_sim=N_SIM_ROUND,
    seed=SEED,
)

print_full_tournament(round_results, round_details, n_sim=N_SIM_ROUND)


Simulation Progress: [##################################################] 100.0%

Tournament Results (round-by-round simulation)

Round of 64
------------------------------------------------------------
  Connecticut vs UT San Antonio  →  Connecticut (991/1000, 99.1%)  
  Iowa St vs Syracuse  →  Syracuse (507/1000, 50.7%)  (fatigue=1.03)
  Maryland vs Murray St  →  Maryland (926/1000, 92.6%)  
  North Carolina vs W Illinois  →  North Carolina (949/1000, 94.9%)  
  Notre Dame vs Fairfield  →  Notre Dame (802/1000, 80.2%)  (fatigue=1.01)
  Ohio St vs Howard  →  Ohio St (985/1000, 98.5%)  
  Illinois vs Colorado  →  Illinois (662/1000, 66.2%)  (fatigue=1.02)
  Vanderbilt vs High Point  →  Vanderbilt (990/1000, 99.0%)  
  UCLA vs Cal Baptist  →  UCLA (1000/1000, 100.0%)  
  Oklahoma St vs Princeton  →  Oklahoma St (591/1000, 59.1%)  (fatigue=1.03)
  Mississippi vs Gonzaga  →  Mississippi (788/1000, 78.8%)  (fatigue=1.01)
  Minnesota vs Green Bay  →  Minnesota (909/1000, 90.9%)  
  Baylor v

In [15]:
# Cell 8b - Round-by-round high-confidence game counts
#
# Count how many games in each round hit >=90% or >=95% win probability.
# These should drop after applying the extra late-round softening layer.
#
# round_details format: list of tuples
#   (team_a, team_b, winner, win_count, p_winner, closeness, fatigue_after)
# p_winner is already the winner's probability (always >= 0.5).

ROUND_ORDER = ["Round of 64", "Round of 32", "Sweet 16", "Elite 8", "Final Four", "Championship"]
ROUND_TO_NUM = {"Round of 64": 1, "Round of 32": 2, "Sweet 16": 3, "Elite 8": 4, "Final Four": 5, "Championship": 6}

print(f"Base temperature (tau) = {params.temperature:.4f}")
print("Effective tau by round:")
for rnd_name in ROUND_ORDER:
    rnd_num = ROUND_TO_NUM[rnd_name]
    eff_tau = params.temperature * W_ROUND_TEMPERATURE_MULTIPLIERS.get(rnd_num, 1.0)
    print(f"  {rnd_name:<18} {eff_tau:.3f}")
print()
print(f"{'Round':<18}  {'Games':>5}  {'≥90%':>5}  {'≥95%':>5}  {'≥90% share':>10}")
print("-" * 52)

total_games = 0
total_hi90  = 0
total_hi95  = 0

for rnd_name in ROUND_ORDER:
    details = round_details.get(rnd_name, [])
    if not details or not isinstance(details, list):
        continue
    n = len(details)
    hi90 = sum(1 for d in details if d[4] >= 0.90)
    hi95 = sum(1 for d in details if d[4] >= 0.95)
    share = hi90 / n if n else 0
    total_games += n
    total_hi90  += hi90
    total_hi95  += hi95
    print(f"{rnd_name:<18}  {n:5d}  {hi90:5d}  {hi95:5d}  {share:10.1%}")

print("-" * 52)
if total_games:
    print(f"{'Total':<18}  {total_games:5d}  {total_hi90:5d}  {total_hi95:5d}  {total_hi90/total_games:.1%}")


Base temperature (tau) = 0.9698
Effective tau by round:
  Round of 64        0.970
  Round of 32        2.000
  Sweet 16           2.200
  Elite 8            2.400
  Final Four         2.550
  Championship       2.550

Round               Games   ≥90%   ≥95%  ≥90% share
----------------------------------------------------
Round of 64            32     17     14       53.1%
Round of 32            16      5      3       31.2%
Sweet 16                8      2      0       25.0%
Elite 8                 4      1      0       25.0%
Final Four              2      0      0        0.0%
Championship            1      0      0        0.0%
----------------------------------------------------
Total                  63     25     17  39.7%
